# Avant le TP9 : travailler avec deux clusters

Le TP9 relie deux clusters : **onprem** (`kafka-1` à `kafka-3`) et **cloud** (`kafka-cloud`). Cette démonstration montre, sans MirrorMaker 2 :

1. comment viser l'un ou l'autre cluster, et les distinguer ;
2. comment comparer un même compteur des deux côtés : l'offset de fin d'un topic ;
3. comment lire un fichier `.properties` ;
4. comment chercher un motif dans les traces d'un conteneur.

Chaque étape montre la version **terminal**, puis la version **Python**. Une étape qui ne fait que lire exécute les deux. Une étape qui modifie le cluster n'en exécute qu'une : l'autre est affichée à côté, **à ne pas lancer en plus**.

Sorties de référence : exécution du 12/11/2026 23:20 UTC sur le lab.

Les adresses, les noms et les valeurs sont définis une seule fois, ici. `BS` vise onprem, `BS_CLOUD` vise cloud ; les deux sont déjà définies dans l'environnement du notebook et du terminal. Dans un terminal de JupyterLab, définissez d'abord : `export TOPIC=demo-tp9 LAB=$FORMATION_DIR/lab`.

In [1]:
import os
import subprocess
import time

from confluent_kafka import KafkaError, KafkaException, Producer, TopicPartition
from confluent_kafka.admin import AdminClient, NewTopic, OffsetSpec

BS = os.environ["BS"]
BS_CLOUD = os.environ["BS_CLOUD"]
LAB = os.environ["FORMATION_DIR"] + "/lab"
# log_level 2 coupe le journal de connexion de librdkafka, qui répéterait
# "Connection refused" tant que le cluster cloud démarre
CLUSTERS = {"onprem": {"bootstrap.servers": BS, "log_level": 2},
            "cloud": {"bootstrap.servers": BS_CLOUD, "log_level": 2}}

TOPIC = "demo-tp9"
PARTITIONS = 3
RF = 3
NUM_MESSAGES = 500
FILE = f"{LAB}/config/kafka/kafka-cloud/server.properties"
SETTINGS = ("listeners", "allow.everyone.if.no.acl.found", "default.replication.factor")
PATTERNS = ("start-cloud", "Kafka Server started")
MAX_WAIT = 90  # secondes pour le démarrage du cluster cloud

## 1. Deux clusters

Le cluster cloud fait partie du profil Compose `mirror`. On le démarre, puis on attend qu'il réponde. Le client Python n'a pas d'équivalent pour démarrer un conteneur : c'est une commande de terminal.

**En terminal**

In [2]:
%%bash
cd $FORMATION_DIR/lab
docker compose --profile mirror up -d kafka-cloud 2>&1

 Volume kafka-lab_kafka-cloud-data Creating 
 Volume kafka-lab_kafka-cloud-data Creating 


 Volume kafka-lab_kafka-cloud-data Created 
 Volume kafka-lab_kafka-cloud-data Created 
 Container k

afka-cloud Creating 


 Container kafka-cloud Created 


 Container kafka-cloud Starting 


 Container kafka-cloud Started 


**En Python** : une demande qui échoue pendant le démarrage est simplement réessayée, jusqu'à `MAX_WAIT` secondes. Le client est gardé dans une variable : détruit avant la réponse, il ferait échouer la demande (`_DESTROY`).

In [3]:
start = time.time()
cloud = AdminClient(CLUSTERS["cloud"])
while True:
    try:
        cloud.describe_cluster(request_timeout=5).result()
        print("cloud répond après %.0f s" % (time.time() - start))
        break
    except KafkaException:
        if time.time() - start > MAX_WAIT:
            raise TimeoutError(f"le cluster cloud ne répond pas après {MAX_WAIT} s")
        time.sleep(2)

cloud répond après 3 s


Distinguer les deux clusters. **En terminal** : seule l'adresse de départ change.

In [4]:
!docker exec kafka-tools kafka-cluster.sh cluster-id --bootstrap-server $BS
!docker exec kafka-tools kafka-cluster.sh cluster-id --bootstrap-server $BS_CLOUD

Cluster ID: W5FmqobtRp-BMcXlZhVKBw


Cluster ID: tF2HiBQuSD6MqpTGGGBVfA


**En Python** : un `AdminClient` par cluster, avec sa configuration.

In [5]:
for name, conf in CLUSTERS.items():
    client = AdminClient(conf)
    description = client.describe_cluster().result()
    print("%-7s cluster_id=%s  noeuds=%s" % (name, description.cluster_id, sorted(n.id for n in description.nodes)))

onprem  cluster_id=W5FmqobtRp-BMcXlZhVKBw  noeuds=[1, 2, 3]
cloud   cluster_id=tF2HiBQuSD6MqpTGGGBVfA  noeuds=[1]


Les deux clusters ont chacun un nœud d'identifiant 1 : seul l'identifiant de cluster les distingue.

## 2. Un même compteur des deux côtés

Le nombre de messages écrits dans un topic est la somme des offsets de fin de ses partitions. On crée `demo-tp9` sur onprem seulement, on y écrit des messages, et on le compte des deux côtés.

Création et envoi, équivalent en terminal (**ne pas lancer en plus**) ; l'outil de test envoie des messages sans clé :

```bash
docker exec kafka-tools kafka-topics.sh --bootstrap-server $BS --create --topic $TOPIC --partitions 3 --replication-factor 3
docker exec kafka-tools kafka-producer-perf-test.sh --topic $TOPIC --num-records 500 --throughput -1 \
  --record-size 4 --producer-props bootstrap.servers=$BS
```

**En Python**

In [6]:
admin = AdminClient(CLUSTERS["onprem"])
try:
    admin.create_topics([NewTopic(TOPIC, num_partitions=PARTITIONS, replication_factor=RF)])[TOPIC].result()
    print(f"{TOPIC} créé sur onprem")
except KafkaException as e:
    if e.args[0].code() != KafkaError.TOPIC_ALREADY_EXISTS:
        raise
    print(f"{TOPIC} existe déjà sur onprem")

producer = Producer({"bootstrap.servers": BS, "log_level": 4})
for i in range(NUM_MESSAGES):
    producer.produce(TOPIC, key=str(i).encode(), value=b"demo")
remaining = producer.flush(10)
if remaining:
    raise RuntimeError(f"{remaining} message(s) non envoyé(s)")

demo-tp9 créé sur onprem


**En terminal** : l'offset de fin de chaque partition, sur chaque cluster. Côté cloud, le topic n'existe pas : l'outil le dit.

In [7]:
!docker exec kafka-tools kafka-get-offsets.sh --bootstrap-server $BS --topic $TOPIC
!docker exec kafka-tools kafka-get-offsets.sh --bootstrap-server $BS_CLOUD --topic $TOPIC

demo-tp9:0:166
demo-tp9:1:178
demo-tp9:2:156


Error occurred: Could not match any topic-partitions with the specified filters


**En Python** : la fonction `end()` fait la somme ; elle renvoie `None` si le topic n'existe pas sur ce cluster.

In [8]:
def end(conf, topic):
    client = AdminClient(conf)
    meta = client.list_topics(timeout=10).topics.get(topic)
    if meta is None or meta.error is not None:
        return None
    offset_requests = {TopicPartition(topic, p): OffsetSpec.latest() for p in meta.partitions}
    return sum(f.result().offset for f in client.list_offsets(offset_requests).values())


print("onprem :", end(CLUSTERS["onprem"], TOPIC), "- cloud :", end(CLUSTERS["cloud"], TOPIC))

onprem : 500 - cloud : None


Au TP, la différence entre les deux côtés sera le **retard de réplication**, en nombre de messages.

## 3. Lire un fichier `.properties`

Les fichiers de configuration du lab sont lisibles depuis le notebook, sous `$FORMATION_DIR/lab` (`FORMATION_DIR` est le dossier de la formation sur votre poste). Exemple avec le nœud cloud.

**En terminal**

In [9]:
!grep -E "^(listeners|allow.everyone.if.no.acl.found|default.replication.factor)=" $FILE

listeners=CONTROLLER://:9093,INTERNAL://:19092,SECURE://:9096
allow.everyone.if.no.acl.found=false
default.replication.factor=1


**En Python** : le fichier devient un dictionnaire ; les lignes vides et les commentaires sont ignorés.

In [10]:
config = {}
with open(FILE) as file:
    for line in file:
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, value = line.split("=", 1)
            config[key.strip()] = value.strip()
print(len(config), "paramètres")
print({key: config.get(key) for key in SETTINGS})

34 paramètres
{'listeners': 'CONTROLLER://:9093,INTERNAL://:19092,SECURE://:9096', 'allow.everyone.if.no.acl.found': 'false', 'default.replication.factor': '1'}


## 4. Chercher dans les traces d'un conteneur

**En terminal**

In [11]:
!docker compose --project-directory $LAB logs kafka-cloud | grep -E "start-cloud|Kafka Server started" | cut -c1-160

kafka-cloud  | [start-cloud] Formatage du stockage (cluster tF2HiBQuSD6MqpTGGGBVfA)
kafka-cloud  | [start-cloud] Lancement de Kafka
kafka-cloud  | [2026-11-12 23:20:09,460] INFO [KafkaRaftServer nodeId=1] Kafka Server started (kafka.server.KafkaRaftServer)


**En Python** : `subprocess.run()` lance la même commande et rend sa sortie ; le filtre se fait en Python.

In [12]:
output = subprocess.run(["docker", "compose", "logs", "kafka-cloud"], cwd=LAB,
                        capture_output=True, text=True, timeout=60)
if output.returncode != 0:
    raise RuntimeError(output.stderr.strip())
logs = output.stdout.splitlines()
print(len(logs), "lignes")
for line in logs:
    if any(pattern in line for pattern in PATTERNS):
        print(line[:160])

2218 lignes
kafka-cloud  | [start-cloud] Formatage du stockage (cluster tF2HiBQuSD6MqpTGGGBVfA)
kafka-cloud  | [start-cloud] Lancement de Kafka
kafka-cloud  | [2026-11-12 23:20:09,460] INFO [KafkaRaftServer nodeId=1] Kafka Server started (kafka.server.KafkaRaftServer)


## 5. Ménage

La démonstration supprime son topic et le cluster cloud, avec son volume : au TP, vous le démarrerez vous-même.

Suppression du topic, équivalent en terminal (**ne pas lancer en plus**) :

```bash
docker exec kafka-tools kafka-topics.sh --bootstrap-server $BS --delete --topic $TOPIC
```

**En Python**

In [13]:
admin = AdminClient(CLUSTERS["onprem"])
try:
    admin.delete_topics([TOPIC])[TOPIC].result()
    print(f"{TOPIC} supprimé")
except KafkaException as e:
    if e.args[0].code() != KafkaError.UNKNOWN_TOPIC_OR_PART:
        raise
    print(f"{TOPIC} absent")

demo-tp9 supprimé


Arrêt et suppression du cluster cloud : commande de terminal seulement.

In [14]:
%%bash
cd $FORMATION_DIR/lab
docker compose --profile mirror rm -sfv kafka-cloud >/dev/null 2>&1
docker volume rm kafka-lab_kafka-cloud-data

kafka-lab_kafka-cloud-data


## Méthode pour le TP9

- Deux clients : `onprem` et `cloud`. Toujours se demander **sur quel cluster** on regarde.
- MirrorMaker 2 n'a pas d'API Python : on le démarre par `docker compose`, on lit ses traces, et on juge son travail **à ses effets** sur le cluster cloud.
- Étape 6.3 : un échec est attendu. Lisez les traces avant de corriger.
- Étape 6.5 : écrivez votre prévision avant de lancer la reprise sur le cloud.